# UrbanLens — train the YOLOv8 detector on a free Colab GPU

This notebook fine-tunes **YOLOv8** on a public (Roboflow) dataset, evaluates it, and exports
`urbanlens_yolov8_v1.pt` that you drop into the repo's `ml/models/` folder. The backend then
loads it automatically (it reads the first `*.pt` in `ml/models`).

### Before you run
1. In Colab: **Runtime > Change runtime type > T4 GPU** (free tier is fine).
2. Get a free **Roboflow API key**: https://app.roboflow.com (account settings).
3. Pick a public dataset. Public data is uneven across the 5 UrbanLens classes, so start
   **pothole-focused** (best data availability) and add the other classes later as you collect more.
   On Roboflow Universe search *pothole detection* — e.g. `misterrobot/potholes`.
4. Fill in the **Config** cell below and run every cell top-to-bottom.


In [ ]:
!pip -q install ultralytics==8.3.40 roboflow opencv-python-headless pyyaml
import torch
print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU'

## 1. Config
Paste your Roboflow details from the dataset page > **Download Dataset > Python** code block
(workspace / project / version), plus your API key. `CLASS_NAMES` must be the classes present in
that dataset, in the same order Roboflow lists them — set the names to the UrbanLens vocabulary
(`pothole`, `garbage`, `damaged_streetlight`, `waterlogging`, `illegal_dumping`). Leave
`CLASS_NAMES = None` to use the dataset's own labels untouched.

In [ ]:
ROBOFLOW_API_KEY = 'PASTE-YOUR-ROBOFLOW-KEY'
WORKSPACE_URL   = 'your-workspace'       # e.g. 'misterrobot'
PROJECT_URL     = 'your-project'         # e.g. 'potholes'
VERSION         = 1

# Rename to UrbanLens classes to match the app's labels, in dataset label order.
# Set to None to keep the dataset's native class names.
CLASS_NAMES = ['pothole']                # single-class pothole demo

MODEL = 'yolov8n.pt'                     # n=s fastest; use 'yolov8s.pt' if you have time
EPOCHS = 60
IMG = 640
BATCH = -1                              # -1 = auto batch for the GPU
SEED = 42
MODEL_VERSION_TAG = 'v1'                # bump when you retrain

## 2. Download the dataset (YOLOv8 format)

In [ ]:
import os
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_URL).project(PROJECT_URL)
version = project.version(VERSION)
dataset = version.download('yolov8')

DATA_DIR = dataset.location
DATA_YAML = os.path.join(DATA_DIR, 'data.yaml')
print('Dataset folder:', DATA_DIR)
assert os.path.exists(DATA_YAML), 'data.yaml not found at ' + DATA_YAML
print(open(DATA_YAML).read())

## 3. Align class names (optional)
Rewrites `data.yaml` with absolute paths and, if `CLASS_NAMES` matches the class count, renames the
classes to the UrbanLens vocabulary so the app displays the right labels.

In [ ]:
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DATA_DIR  # absolute so ultralytics resolves train/val
if isinstance(cfg.get('names'), dict):
    names = [cfg['names'][k] for k in sorted(cfg['names'])]
else:
    names = list(cfg.get('names', []))

if CLASS_NAMES:
    if len(CLASS_NAMES) != len(names):
        print('WARNING: CLASS_NAMES has', len(CLASS_NAMES), 'but dataset has', len(names), names)
        print('        Edit CLASS_NAMES to match, or set it to None. Keeping native names for now.')
    else:
        names = list(CLASS_NAMES)
        print('Renamed classes to:', names)

cfg['names'] = names
cfg['nc'] = len(names)
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(DATA_YAML).read())

## 4. Train

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG,
    batch=BATCH,
    device=0,
    seed=SEED,
    project='/content/runs',
    name='urbanlens',
    exist_ok=True,
    pretrained=True,
    optimizer='SGD',
    patience=20,
)

## 5. Evaluate + capture metrics

In [ ]:
best = model.trainer.best  # path to best.pt
metrics = YOLO(str(best)).val(data=DATA_YAML, device=0, verbose=True)

mAP50 = float(metrics.box.map50)
mAP50_95 = float(metrics.box.map)
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
class_names = metrics.names.values() if isinstance(metrics.names, dict) else metrics.names

print('== RESULTS ==')
print('  mAP@50   =', round(mAP50, 3))
print('  mAP@50-95=', round(mAP50_95, 3))
print('  precision=', round(precision, 3))
print('  recall   =', round(recall, 3))
print('  classes  =', list(class_names))
assert mAP50 > 0, 'Zero mAP - check dataset labels / class alignment before exporting.'

## 6. Sample predictions (sanity check on real images)

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

val_imgs = glob.glob(os.path.join(DATA_DIR, 'valid/images/*'))[:4]
if not val_imgs:
    val_imgs = glob.glob(os.path.join(DATA_DIR, '*/images/*'))[:4]
if val_imgs:
    fig, axes = plt.subplots(1, len(val_imgs), figsize=(5 * len(val_imgs), 5))
    axes = [axes] if len(val_imgs) == 1 else axes
    for ax, p in zip(axes, val_imgs):
        r = YOLO(str(best))(p, conf=0.25, device=0)[0]
        ax.imshow(Image.fromarray(r.plot()[:, :, ::-1]))
        ax.set_title(os.path.basename(p)); ax.axis('off')
    plt.tight_layout(); plt.show()
print('If the boxes look right, continue to export.')

## 7. Export the weights + write a model card

In [ ]:
import shutil
OUT_PT = '/content/urbanlens_yolov8_' + MODEL_VERSION_TAG + '.pt'
shutil.copy(str(best), OUT_PT)
print('Saved:', OUT_PT)

card = '\n'.join([
    '# YOLOv8-UrbanLens ' + MODEL_VERSION_TAG,
    '',
    '## Metrics (auto-filled from this Colab run)',
    '| Metric | Value |',
    '| --- | --- |',
    '| mAP@50 | ' + format(mAP50, '.3f') + ' |',
    '| mAP@50-95 | ' + format(mAP50_95, '.3f') + ' |',
    '| Precision | ' + format(precision, '.3f') + ' |',
    '| Recall | ' + format(recall, '.3f') + ' |',
    '',
    'Classes: ' + str(list(class_names)),
    '',
    '## Training data',
    '- Source: Roboflow ' + WORKSPACE_URL + '/' + PROJECT_URL + ' v' + str(VERSION) + ' (public dataset)',
    '- Base model: ' + MODEL + ', epochs=' + str(EPOCHS) + ', imgsz=' + str(IMG) + ', seed=' + str(SEED),
    '- NOTE: record the dataset license/attribution here before publishing.',
    '',
    '## Limitations',
    '- Trained on public data; not Navi Mumbai specific and not yet multi-class per issue.',
    '- Performance degrades in heavy rain / low light. Decision-support only.',
    '',
])
with open('/content/model_card.md', 'w') as f:
    f.write(card)
print(card)

## 8. Download artifacts
Click the downloaded files, then follow the **Deploy locally** steps in `ml/README.md`.

In [ ]:
from google.colab import files
files.download(OUT_PT)
files.download('/content/model_card.md')
!cd /content && zip -qr urbanlens_run.zip runs model_card.md && echo 'run zip ready at /content/urbanlens_run.zip'